In [1]:
# Uncomment and run this cell if you're on Colab or Kaggle
# !git clone https://github.com/nlp-with-transformers/notebooks.git
# %cd notebooks
# from install import *
# install_requirements(is_chapter10=True)

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [2]:
# hide
from utils import *
setup_chapter()

Using transformers v5.14.1
Using datasets v5.0.0


# Training Transformers from Scratch

> **Note:** In this chapter a large dataset and the script to train a large language model on a distributed infrastructure are built. As such not all the steps in this notebook are executable on platforms such as Colab or Kaggle. Either downscale the steps at critical points or use this notebook as an inspiration when building a script for distributed training.

## Large Datasets and Where to Find Them

### Challenges of Building a Large-Scale Corpus

In [3]:
#hide_output

# comparamos dos generaciones de GPT.
# Crea dos pipelines de generación de texto:
    # generation_gpt → carga OpenAI GPT, el GPT original (GPT-1).
    # generation_gpt2 → carga GPT-2.

# En ambos casos "text-generation" le dice a Transformers qué tarea quieres realizar. 
# pipeline() se ocupa de cargar automáticamente el modelo y el tokenizer apropiados.

from transformers import pipeline, set_seed

generation_gpt = pipeline("text-generation", model="openai-gpt")
generation_gpt2 = pipeline("text-generation", model="gpt2")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
# Función para contar el número de parámetros en cada modelo

def model_size(model):
    return sum(t.numel() for t in model.parameters())

# model.parameters():Devuelve todos los tensores de parámetros del modelo: 
    # matrices de pesos, biases, embeddings, etc.
# numel() -> number of elements.
    # si t es un tensor de de shape (768, 768) -> t.numel(589924)
    

print(f"GPT  size: {model_size(generation_gpt.model)/1000**2:.1f}M parameters")
print(f"GPT2 size: {model_size(generation_gpt2.model)/1000**2:.1f}M parameters")
# lo divide por 1.000.000 porque se expresa en millones (M) de parámetros con un  decimal

GPT  size: 116.5M parameters
GPT2 size: 124.4M parameters


In [5]:
# hide
set_seed(1)

In [6]:
# Función para comparar ambos modelos

def enum_pipeline_ouputs(pipe, prompt, num_return_sequences):
    out = pipe(prompt, num_return_sequences=num_return_sequences,
               clean_up_tokenization_spaces=True)
    return "\n".join(f"{i+1}." + s["generated_text"] for i, s in enumerate(out))

prompt = "\nWhen they came back"
print("GPT completions:\n" + enum_pipeline_ouputs(generation_gpt, prompt, 3))
print("")
print("GPT-2 completions:\n" + enum_pipeline_ouputs(generation_gpt2, prompt, 3))

GPT completions:
1.
When they came backto their cottage and saw it was empty .
 " my god , " she said . " he 's gone . "
 " no . he 's not , " said the man in the black , velvet coat . " he 's gone ,
and we are going to get him back . "
 " no ! " she said again . " no , i ca n't . please . "
 " do n't you see ? " said the man , a little louder . " he 's dead . he 's gone
. "
 " no ! " she cried out . " the man was here and he did n't come back . he never
came back . "
 the man looked at her again , and then he looked at the man in the velvet coat
.
 " you must go , " he said . " take the little boy . he must go . "
 " no , " said the man in the velvet coat . " i will go . you have to leave me .
"
 " do n't you see ? " she cried out . " he 's gone ! "
 it was as if someone had turned the sound off on the radio . the man in the
velvet coat was silent . the man in the brown coat was silent .
 michael was in the kitchen .
 "
2.
When they came backshe was a full foot taller than him and had

### Building a Custom Code Dataset


#### Creating a dataset with Google BigQuery

#sidebar To Filter the Noise or Not?

### Working with Large Datasets

#### Memory mapping

> **Note:** The following code block assumes that you have downloaded the BigQuery dataset to a folder called `codeparrot`. We suggest skipping this step since it will unpack the compressed files and require ~180GB of disk space. This code is just for demonstration purposes and you can just continue below with the streamed dataset which will not consume that much disk space.

In [ ]:
#hide_output
#Memory mapping

from datasets import load_dataset, DownloadConfig
# load_dataset -> construye/carga un dataset de Hugging Face Datasets. 
# DownloadConfig -> permite decirle cómo queremos que gestione la descarga y los archivos temporales.

download_config = DownloadConfig(delete_extracted=True)
# (delete_extracted=True -> borramos los archivos que no necesitamos lo antes posible
# le dices que elimine esos archivos extraídos después de utilizarlos.

dataset = load_dataset("./codeparrot", split="train",
                       download_config=download_config)
# cargamos desde ./codeparrot el dataset que debería estar descargado antes (no lo hemos hecho, 
# pq lo haremos de otra manera después)
# Busca el dataset en la carpeta local codeparrot relativa al directorio donde estoy trabajando.
# "." -> directorio actual
# split="train" -> cargar la partición de entrenamiento.

In [ ]:
import psutil, os
# os -> permite consultar información del sistema y de archivos.
# psutil -> permite consultar, entre otras cosas, cuánta RAM está utilizando el proceso de Python.

print(f"Number of python files code in dataset : {len(dataset)}")
# len(dataset) devuelve el número de ejemplos del dataset. 
# En CodeParrot, cada ejemplo corresponde esencialmente a un archivo de código Python.


ds_size = sum(os.stat(f["filename"]).st_size for f in dataset.cache_files)
# ds_size -> es el tamaño total que ocupa en disco el dataset preparado/cacheado.
# dataset.cache_files -> contiene información sobre los archivos de caché que Hugging Face Datasets 
# ha creado en disco para representar el dataset.
# f["filename"] -> obtiene su ruta
# os.stat(...).st_size -> Obtiene su tamaño en bytes.

# os.stat.st_size is expressed in bytes, so we convert to GB
print(f"Dataset size (cache file) : {ds_size / 2**30:.2f} GB")
# Para convertir en GB Dvidimos por 2^30 (1.073.741.824)


# Process.memory_info is expressed in bytes, so we convert to MB
print(f"RAM used: {psutil.Process(os.getpid()).memory_info().rss >> 20} MB")
# os.getpid() obtiene el identificador del proceso de Python que está ejecutando tu notebook.
# psutil.Process(...) accede a ese proceso
# .memory_info().rss -> obtiene su RSS (Resident Set Size): aproximadamente, cuánta memoria física RAM está 
# ocupando actualmente ese proceso.-> Devuelve datos en bytes
# >>20 Desplaza los bits n20 posiciones hacia la dereccha  equivale a dividir por 2^20 (1.048.576) -> Para pasarlo a MB
# Lo mismo que: rss / 2**20

#### Streaming

In [ ]:
# hide_output

from datasets import load_dataset

streamed_dataset = load_dataset('./codeparrot', split="train", streaming=True)

#streaming=True -> No me descargues y prepares todo el dataset. Dame los datos conforme los vaya necesitando

In [ ]:
iterator = iter(streamed_dataset)

print(dataset[0] == next(iterator))
print(dataset[1] == next(iterator))

In [8]:
from datasets import load_dataset

remote_dataset = load_dataset('transformersbook/codeparrot', split="train",
                              streaming=True)

Resolving data files:   0%|          | 0/184 [00:00<?, ?it/s]

### Adding Datasets to the Hugging Face Hub

- Esta sección del libro muestra cómo los autores:
    - 1) separaron train/validation,
    - 2) crearon repositorios en Hugging Face Hub,
    - 3) subieron los .json.gz,
    - 4) y luego los consumieron por streaming.

- No lo reproduzco porque el dataset ya está publicado en HF Hub.


## Building a Tokenizer

Ahora el objetivo es estudiar **cómo construir un tokenizer adecuado para código**, en lugar de utilizar directamente el de un modelo existente.

La función permite ver cómo un tokenizer divide un texto:

```text
texto
  ↓
tokenizer
  ↓
input_ids
  ↓
decodificar cada ID
  ↓
tokens visibles
```

```

La idea importante es que **una misma cadena puede dividirse de formas muy diferentes según el tokenizer**.

Un tokenizer entrenado principalmente con lenguaje natural puede ser poco eficiente para código, donde aparecen elementos específicos como:

```text
def
self
==
!=
numpy
nombres de funciones
operadores
indentación
```

Por eso el flujo será:

```text
Corpus CodeParrot
      ↓
Entrenar tokenizer sobre código
      ↓
Vocabulario específico para código
      ↓
Entrenar el Transformer desde cero
```

Esta sección muestra, por tanto, **de dónde sale el tokenizer que posteriormente cargamos con `AutoTokenizer.from_pretrained(...)`**.

In [9]:
# hide_output
from transformers import AutoTokenizer

def tok_list(tokenizer, string):
    input_ids = tokenizer(string, add_special_tokens=False)["input_ids"]
    return [tokenizer.decode(tok) for tok in input_ids]

tokenizer_T5 = AutoTokenizer.from_pretrained("t5-base")
tokenizer_camembert = AutoTokenizer.from_pretrained("camembert-base")

In [10]:
print(f'T5 tokens for "sex": {tok_list(tokenizer_T5,"sex")}')
print(f'CamemBERT tokens for "being": {tok_list(tokenizer_camembert,"being")}')

T5 tokens for "sex": ['', 's', 'ex']
CamemBERT tokens for "being": ['be', 'ing']


La calidad de un tokenizer depende del corpus sobre el que se haya entrenado.

Por eso, si ahora quieres entrenar un Transformer especializado en código Python, no es ideal coger sin más un tokenizer entrenado sobre inglés general, francés, etc.

### The Tokenizer Model

### Measuring Tokenizer Performance

### A Tokenizer for Python 

In [11]:
from transformers import AutoTokenizer

python_code = r"""def say_hello():
    print("Hello, World!")
# Print it
say_hello()
"""
tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(tokenizer(python_code).tokens())

['def', 'Ġsay', '_', 'hello', '():', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġprint', '("',
'Hello', ',', 'ĠWorld', '!"', ')', 'Ċ', '#', 'ĠPrint', 'Ġit', 'Ċ', 'say', '_',
'hello', '()', 'Ċ']


In [12]:
print(tokenizer.backend_tokenizer.normalizer)

None


In [13]:
print(tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(python_code))

[('def', (0, 3)), ('Ġsay', (3, 7)), ('_', (7, 8)), ('hello', (8, 13)), ('():',
(13, 16)), ('ĊĠĠĠ', (16, 20)), ('Ġprint', (20, 26)), ('("', (26, 28)), ('Hello',
(28, 33)), (',', (33, 34)), ('ĠWorld', (34, 40)), ('!")', (40, 43)), ('Ċ', (43,
44)), ('#', (44, 45)), ('ĠPrint', (45, 51)), ('Ġit', (51, 54)), ('Ċ', (54, 55)),
('say', (55, 58)), ('_', (58, 59)), ('hello', (59, 64)), ('()', (64, 66)), ('Ċ',
(66, 67))]


Este código está enseñando cómo UTF-8 representa distintos caracteres mediante bytes, y la diferencia entre "a" y "€" es muy ilustrativa.

**idea: trabajar anivel de bytes usando UTF-8**

In [14]:
# Creamos dos strings unicode: Ya no es necesario poner "u" delante pq los string en python3 son unicode
a = u"a"
e = u"€"

# convertimos el carácter "a" a su representación UTF-8 (unicode):
byte = ord(a.encode("utf-8")) 
# a.encode("utf-8") -> bytes -> b'a'
# ord(a.encode(b'a') -> 97 -> valor numérico del byte
print(f'`{a}` is encoded as `{a.encode("utf-8")}` with a single byte: {byte}')

# convertimos el carácter "€" a su representación UTF-8 (unicode):
byte = [ord(chr(i)) for i in e.encode("utf-8")]
# e.encode("utf-8") -> utf-8 necesita 3 bytes para representar € -> b'\xe2\x82\xac'
# for i in e.encode("utf-8") -> Iteramos sobre el objeto bytes que contiene 3 bytes 
# cuando iteramos sobre un objeto byte -> Python devuelve un entero sin necesidad de ord() 
# chr(i) -> Volvemos a para a caracteres de byte
# ord(chr(i))- > Pasamos nuevamente a número entero

# se podría haber hecho:
# byte = list(e.encode("utf-8"))

print(f'`{e}` is encoded as `{e.encode("utf-8")}` with three bytes: {byte}')

`a` is encoded as `b'a'` with a single byte: 97
`€` is encoded as `b'\xe2\x82\xac'` with three bytes: [226, 130, 172]


```text
Texto Unicode
      │
      ▼
UTF-8
      │
      ▼
256 posibles bytes
      │
      ▼
Mapeo a 256 símbolos imprimibles
      │
      ▼
BPE une patrones frecuentes
      │
      ▼
Tokens
      │
      ▼
input_ids
      │
      ▼
Transformer
```

In [15]:
# añado esta función porque from transformers.models.gpt2.tokenization_gpt2 import bytes_to_unicode ya no existe

def bytes_to_unicode():
    """
    Devuelve el mapeo reversible de los 256 valores de byte
    a caracteres Unicode imprimibles usado por GPT-2.
    """
    bs = list(range(ord("!"), ord("~") + 1)) + \
         list(range(ord("¡"), ord("¬") + 1)) + \
         list(range(ord("®"), ord("ÿ") + 1))

    cs = bs[:]
    n = 0

    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1

    cs = [chr(n) for n in cs]

    return dict(zip(bs, cs))

In [16]:
#from transformers.models.gpt2.tokenization_gpt2 import bytes_to_unicode
# Importas la función que utiliza el tokenizer de GPT-2 para construir ese mapeo.


byte_to_unicode_map = bytes_to_unicode()
# crea un diccionario byte (int) → carácter Unicode
# {
#    33: '!',
#    34: '"',
#    ...
#    32: 'Ġ',
#    ...
# }

# Hay 256 entradas, una por cada posible valor de un byte (0–255).
# Por ejemplo, el espacio tiene byte:
# ord(" ")
# 32
# y GPT-2 lo representa inyternamente cimo 32 -> Ġ


unicode_to_byte_map = dict((v, k) for k, v in byte_to_unicode_map.items())
# Hacemos diccionario inverso -> Para cada pareja k, v, crea una nueva pareja v, k.
# aquí será por ejemplo Ġ → 32

base_vocab = list(unicode_to_byte_map.keys())
# Hacemos una con los caracteres (claves) que representas los 256 bytes

print(f'Size of our base vocabulary: {len(base_vocab)}')
print(f'First element: `{base_vocab[0]}`, last element: `{base_vocab[-1]}`')

Size of our base vocabulary: 256
First element: `!`, last element: `Ń`


In [17]:
# hide_input
#id unicode_mapping
#caption Examples of character mappings in BPE
#hide_input
import pandas as pd
# from transformers.models.gpt2.tokenization_gpt2 import bytes_to_unicode

byte_to_unicode_map = bytes_to_unicode()
unicode_to_byte_map = dict((v, k) for k, v in byte_to_unicode_map.items())
base_vocab = list(unicode_to_byte_map.keys())

examples = [
    ['Regular characters', '`a` and `?`', f'{ord("a")} and {ord("?")}' , f'`{byte_to_unicode_map[ord("a")]}` and `{byte_to_unicode_map[ord("?")]}`'],
    ['Nonprintable control character (carriage return)', '`U+000D`', f'13', f'`{byte_to_unicode_map[13]}`'],
    ['A space', '` `', f'{ord(" ")}', f'`{byte_to_unicode_map[ord(" ")]}`'],
    ['A nonbreakable space', '`\\xa0`', '160', f'`{byte_to_unicode_map[ord(chr(160))]}`'],
    ['A newline character', '`\\n`', '10', f'`{byte_to_unicode_map[ord(chr(10))]}`'],
]

pd.DataFrame(examples, columns = ['Description', 'Character', 'Bytes', 'Mapped bytes'])

,Description,Character,Bytes,Mapped bytes
0,Regular characters,`a` and `?`,97 and 63,`a` and `?`
1,Nonprintable control character (carriage return),`U+000D`,13,`č`
2,A space,` `,32,`Ġ`
3,A nonbreakable space,`\xa0`,160,`ł`
4,A newline character,`\n`,10,`Ċ`


In [18]:
print(tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(python_code))

[('def', (0, 3)), ('Ġsay', (3, 7)), ('_', (7, 8)), ('hello', (8, 13)), ('():',
(13, 16)), ('ĊĠĠĠ', (16, 20)), ('Ġprint', (20, 26)), ('("', (26, 28)), ('Hello',
(28, 33)), (',', (33, 34)), ('ĠWorld', (34, 40)), ('!")', (40, 43)), ('Ċ', (43,
44)), ('#', (44, 45)), ('ĠPrint', (45, 51)), ('Ġit', (51, 54)), ('Ċ', (54, 55)),
('say', (55, 58)), ('_', (58, 59)), ('hello', (59, 64)), ('()', (64, 66)), ('Ċ',
(66, 67))]


In [19]:
print(f"Size of the vocabulary: {len(tokenizer)}")

Size of the vocabulary: 50257


- Vocabulario base: 256 valores de bytes
- 50.000 tokens adicionales creados para fusionar repetidamente los tokens con mayor ocurrencia -> Compuesto por varios bytes
- Un caracter especial para representar las fronteras del documento

In [20]:
print(tokenizer(python_code).tokens())

['def', 'Ġsay', '_', 'hello', '():', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġprint', '("',
'Hello', ',', 'ĠWorld', '!"', ')', 'Ċ', '#', 'ĠPrint', 'Ġit', 'Ċ', 'say', '_',
'hello', '()', 'Ċ']


### Training a Tokenizer

In [21]:
# Esta celda quiere ver cuáles son los tokens más largos que contiene el vocabulario del tokenizer.

tokens = sorted(tokenizer.vocab.items(), key=lambda x: len(x[0]), reverse=True)
# tokenizer.vocab es un diccionario token -> token_id

# {
#     "hello": 1234,
#     "Ġpython": 5678,
#     "ing": 910,
#     ...
# }

# Con .items() obtienes parejas:
# ("hello", 1234)
# ("Ġpython", 5678)
# ...

# x -> cada pareja 
# x[0] -> "hello"
# len(x[0]) ->5
# key=lambda x: len(x[0]) -> ordena cada pareja según la longitud del token
# key es la clave del ordenado -> sorted(lista, key=...)
# reverse=True -> de mayr a menor

print([f'{tokenizer.convert_tokens_to_string([t])}' for t, _ in tokens[:8]]);
#  for t, _ in tokens[:8]] -> desempaquetamos la tupla, sólo nos interesa el token (t), desechamos el token_id (_)
# t es un string pero .convert_tokens_to_string() espera una lista de tokens por eso ->[t] -> lista q contiene ese string
# .convert_tokens_to_string() -> Convierte la representación interna del tokenizer a texto normal.

# «De los 8 tokens más largos, coge cada token (ignorando su ID), conviértelo de la representación interna del tokenizer 
# a texto legible y muestra la lista resultante.»

['ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ', '
=================================================================', '
----------------------------------------------------------------',
'ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ',
'................................................................',
'================================================================',
'----------------------------------------------------------------',
'________________________________________________________________']


In [22]:
tokens = sorted(tokenizer.vocab.items(), key=lambda x: x[1], reverse=True)
# ordenamos siguiendo el token_id como criterio -> Los del final -> los merges menos frecuentes

print([f'{tokenizer.convert_tokens_to_string([t])}' for t, _ in tokens[:12]]);

['<|endoftext|>', ' gazed', ' informants', ' Collider', ' regress', 'ominated',
' amplification', 'Compar', '…."', ' (/', 'Commission', ' Hitman']


¿Qué quiere demostrar?

Que algunos de esos últimos tokens son demasiado específicos.

Por ejemplo, menciona nombres propios como Hitman y Commission. Si el tokenizer les concede un token propio:

"Hitman" → 1 token

l modelo necesitará además un embedding específico para Hitman.

Esto tiene un coste porque la matriz de embeddings contiene: vocab_size × embedding_dimension

estamos dedicando parámetros del modelo a tokens que quizá aparecen poquísimo.

Y de ahí viene una observación muy interesante de Tunstall:

**La aparición de muchos tokens extremadamente específicos puede indicar que el vocabulario elegido es demasiado grande o que el corpus contiene elementos idiosincráticos**

Ahora vamos a dejar de usar gpt2 y vamos a netrenar un tokenizer en código Python

In [23]:
#hide_output
# Aquí entrena de verdad el tokenizer nuevo. 
# La idea general es:
# CodeParrot en streaming (Corpus)->leer 100k docs ->entregarlo por batches-> train_new_from_iterator() -> 
# -> BPE aprende nuevos merges -> Tokenizer nuevo de 12500 tokens


from tqdm.auto import tqdm
# Para msotrar barra de progreso

length = 100000 
# necesitaremos 100.000 documentos / 1–2 GB

dataset_name = 'transformersbook/codeparrot-train'
dataset = load_dataset(dataset_name, split="train", streaming=True)
iter_dataset = iter(dataset)
# Aquí cargamos CodeParrot mediante streaming y creas el iterador que irá entregando ejemplos uno detrás de otro.

def batch_iterator(batch_size=10):
    for _ in tqdm(range(0, length, batch_size)):
        # range(0,100000,10) -> se ejecutan 100k iteraciones de 10 en 10 
        yield [next(iter_dataset)['content'] for _ in range(batch_size)]
        # en cada una de esas iteraciones. hace 10 veces esto:
            # next(iter_dataset) -> obtiene el siguiente documento de CodeParrot
            # y selecciona únicamente ['content'] -> q contiene el código
            # yield lo va generando secuencialmente

new_tokenizer = tokenizer.train_new_from_iterator(batch_iterator(), 
                                                  vocab_size=12500,
                                                  initial_alphabet=base_vocab)
# Aquí se realiza el entrenamiento

# tokenizer -> el tokenizer de GPT-2 que están utilizando como plantilla de arquitectura/configuración.
# train_new_from_iterator() -> conserva el tipo de tokenizer y vuelve a entrenar su vocabulario sobre tu nuevo corpus.
# vocab_size=12500 -> Construye un vocabulario final de aproximadamente 12.500 tokens
# initial_alphabet=base_vocab -> Le dices que el punto de partida debe contener los 256 símbolos correspondientes a los 256 posibles bytes.

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

In [28]:
tokens = sorted(new_tokenizer.vocab.items(), key=lambda x: x[1], reverse=False)
print([f'{tokenizer.convert_tokens_to_string([t])}' for t, _ in tokens[257:280]]);

['  ', '    ', '   ', '        ', 'se', 'in', '       ', 're', 'on', 'te', '\n
', '\n        ', 'or', 'st', 'de', '\n   ', 'th', 'le', ' =', 'lf', 'self',
'me', 'al']


In [29]:
print([f'{new_tokenizer.convert_tokens_to_string([t])}' for t,_ in tokens[-12:]]);

[' capt', ' embedded', ' regarding', 'Bundle', '355', ' recv', ' dmp', ' vault',
' Mongo', ' possibly', 'implementation', 'Matches']


In [27]:
# tokenizamos nuestro ejemplo
print(new_tokenizer(python_code).tokens())

['def', 'Ġs', 'ay', '_', 'hello', '():', 'ĊĠĠĠ', 'Ġprint', '("', 'Hello', ',',
'ĠWor', 'ld', '!")', 'Ċ', '#', 'ĠPrint', 'Ġit', 'Ċ', 's', 'ay', '_', 'hello',
'()', 'Ċ']


In [30]:
import keyword

print(f'There are in total {len(keyword.kwlist)} Python keywords.')
for keyw in keyword.kwlist:
    if keyw not in new_tokenizer.vocab:
        print(f'No, keyword `{keyw}` is not in the vocabulary')

There are in total 35 Python keywords.
No, keyword `await` is not in the vocabulary
No, keyword `finally` is not in the vocabulary
No, keyword `nonlocal` is not in the vocabulary


In [31]:
# hide_output
# Vanos a entrenar con una con un dataset mayor
length = 200000
new_tokenizer_larger = tokenizer.train_new_from_iterator(batch_iterator(),
    vocab_size=32768, initial_alphabet=base_vocab)

  0%|          | 0/20000 [00:00<?, ?it/s]

In [33]:
tokens = sorted(new_tokenizer_larger.vocab.items(), key=lambda x: x[1],
                reverse=False)
print([f'{tokenizer.convert_tokens_to_string([t])}' for t, _ in tokens[-12:]]);

[" '<?", 'Functional', ' Images', 'encoders', ' bibrec', ' OPTIONAL', '
rdclass', 'SocketAddressTag', '资金', 'DEPLOYMENT', '经纪公司代码', ")'],"]


In [34]:
print(new_tokenizer_larger(python_code).tokens())

['def', 'Ġsay', '_', 'hello', '():', 'ĊĠĠĠ', 'Ġprint', '("', 'Hello', ',',
'ĠWorld', '!")', 'Ċ', '#', 'ĠPrint', 'Ġit', 'Ċ', 'say', '_', 'hello', '()', 'Ċ']


In [35]:
for keyw in keyword.kwlist:
    if keyw not in new_tokenizer_larger.vocab:
        print(f'No, keyword `{keyw}` is not in the vocabulary')

No, keyword `nonlocal` is not in the vocabulary


### Saving a Custom Tokenizer on the Hub

In [37]:
#hide_output
model_ckpt = "codeparrot"
#org = "transformersbook"
new_tokenizer_larger.push_to_hub(model_ckpt)

CommitInfo(commit_url='https://huggingface.co/srmjfba/codeparrot/commit/802acca71682491b4c383ce51cf11152d5357f4a', commit_message='Upload tokenizer', commit_description='', oid='802acca71682491b4c383ce51cf11152d5357f4a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/srmjfba/codeparrot', endpoint='https://huggingface.co', repo_type='model', repo_id='srmjfba/codeparrot'), pr_revision=None, pr_num=None)

In [12]:
# Cargo mi nuevo tokenizador desde HF
from transformers import AutoTokenizer

reloaded_tokenizer = AutoTokenizer.from_pretrained("srmjfba/codeparrot")
print(reloaded_tokenizer(python_code).tokens())

['def', 'Ġsay', '_', 'hello', '():', 'ĊĠĠĠ', 'Ġprint', '("', 'Hello', ',',
'ĠWorld', '!")', 'Ċ', '#', 'ĠPrint', 'Ġit', 'Ċ', 'say', '_', 'hello', '()', 'Ċ']


In [6]:
#hide_output
# Hacemos lo mismo pero con el tokenizador pequeño
new_tokenizer.push_to_hub(model_ckpt+ "-small-vocabulary")

NameError: name 'new_tokenizer' is not defined

## Training a Model from Scratch

### A Tale of Pretraining Objectives

<img alt="Code snippet" caption="An example of a Python function that could be found in our dataset" src="images/chapter10_code-snippet.png" id="code-snippet"/>

#### Causal language modeling

<img alt="CLM pretraining" caption="In causal language modeling, the future tokens are masked and the model has to predict them; typically a decoder model such as GPT is used for such a task" src="images/chapter10_pretraining-clm.png" id="pretraining-clm"/>

#### Masked language modeling

<img alt="MLM pretraining" caption="In masked language modeling some of the input tokens are either masked or replaced, and the model's task is to predict the original tokens; this is the architecture underlying the encoder branch of transformer models" src="images/chapter10_pretraining-mlm.png" id="pretraining-mlm"/>

#### Sequence-to-sequence training

<img alt="Seq2seq pretraining" caption="Using an encoder-decoder architecture for a sequence-to-sequence task where the inputs are split into comment/code pairs using heuristics: the model gets one element as input and needs to generate the other one" src="images/chapter10_pretraining-seq2seq.png" id="pretraining-seq2seq"/>

| Quiero que el modelo aprenda a... | Objetivo | Arquitectura típica |
|---|---|---|
| Predecir lo que viene después | **Causal LM** | Decoder (GPT) |
| Reconstruir partes ocultas | **Masked LM** | Encoder (BERT) |
| Transformar una secuencia en otra | **Seq2Seq** | Encoder-decoder (T5) |


### Initializing the Model

> **NOTE**: In the following code block, a large GPT-2 checkpoint is loaded into memory. On platforms like Colab and Kaggle, this can cause the instance to crash due to insufficient RAM or GPU memory. You can still run the example if you use the small checkpoint by replacing the configuration with `config = AutoConfig.from_pretrained("gpt2", vocab_size=len(tokenizer))`.

In [7]:
#hide_output
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
# AutoTokenizer       → convierte texto ↔ tokens/IDs
# AutoConfig          → define la arquitectura del Transformer
# AutoModelForCausalLM → construye el modelo para Causal Language Modeling

tokenizer = AutoTokenizer.from_pretrained("srmjfba/codeparrot") # tokenizer que entrené ayer
# cod python -> nuestro tkenizer -> input_ids
config = AutoConfig.from_pretrained("gpt2-xl", vocab_size=len(tokenizer))
# Aquí NO estás cargando los pesos preentrenados de GPT-2 XL.
# Estás cargando solamente su configuración arquitectónica-> num de capaas, dim de embeddings, num de attention heads, context length...
# vocab_size=len(tokenizer) -> porque GPT-2 XL original estaba diseñado para su vocabulario, 
# pero nosotros hemos entrenado nuestro propio vocabulario para Python.

model = AutoModelForCausalLM.from_config(config)
# Aquí ya no usamos "from pretrained()"
# Estamos diciendo "Construye un modelo con esta arquitectura, pero inicializa sus parámetros desde cero." -> Ya lo entrenaré con CodeParrot
# ForCausalLM -> añade/selecciona la cabeza necesaria para el objetivo que acabamos de estudiar:
    # tokens anteriores - Transformer -> probabilidad del siguiente token

In [17]:
print(f'GPT-2 (xl) size: {model_size(model)/1000**2:.1f}M parameters')
# Cuenta los millones de parámetros -> 1000^2= 1.000.000

GPT-2 (xl) size: 1529.6M parameters


In [23]:
#hide_output
model.save_pretrained("models/" + model_ckpt, push_to_hub=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Upload 0 LFS files: 0it [00:00, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


In [24]:
# misma idea que antes, pero ahora uso la arquitectura de GPT-2 normal, no GPT-2 XL. 
# model_small tiene pesos recién inicializados

tokenizer = AutoTokenizer.from_pretrained("srmjfba/codeparrot")
config_small = AutoConfig.from_pretrained("gpt2", vocab_size=len(tokenizer))
model_small = AutoModelForCausalLM.from_config(config_small)


config.json:   0%|          | 0.00/985 [00:00<?, ?B/s]

In [25]:
print(f'GPT-2 size: {model_size(model_small)/1000**2:.1f}M parameters')

GPT-2 size: 111.0M parameters


In [26]:
#hide_output
model_small.save_pretrained("models/" + model_ckpt + "-small", push_to_hub=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/444M [00:00<?, ?B/s]

### Implementing the Dataloader

<img alt="Preprocessing for CLM" caption="Preparing sequences of varying length for causal language modeling by concatenating several tokenized examples with an EOS token  before chunking them" src="images/chapter10_preprocessing-clm.png" id="preprocessing-clm"/>

**Los límites de los inputs de entrenamiento NO tienen por qué coincidir con los límites de los archivos.**

```text
archivo A       archivo B                 archivo C
████████|EOS|████████████████|EOS|██████████████████
|----------1024----------|----------1024----------|
        Input 1                   Input 2
```

Así, `Input 1` puede contener:

- El archivo A entero.
- Su token `EOS`.
- Una parte del archivo B.

Y `Input 2` continúa con el resto del archivo B.

El útlmo input podría quedar con menos de 1024 caracteres -> Podríamos rellenarlo de padding pero no vale la pena. Lo descartamos.

Por qué la idea de las 100 secuencias? (100 secuencias x 1024 tokens = 102.400 tokens)

Porque si finalmente perdemos una secuancias la pérdida relativa será del 1%

Otro Problema **¿cuánto texto necesito para esoso 102.400 tokens?**

el dataset originalmente contiene texto, no tokens. Ahí aparece:

input_characters = (number_of_sequences * sequence_length * characters_per_token)

donde:

number_of_sequences = 100

sequence_length = 1024

characters_per_token = ? -> ¿Cuántos caracteres de texto representa, en promedio, cada token de nuestro tokenizer?

Por ejemplo, imaginemos:

"def calculate_total"

tiene unos 19 caracteres y el tokenizer produce, imaginemos, 6 tokens.

Entonces:

19 caracteres / 6 tokens
≈ 3,17 caracteres/token

Hay que calcular esa media sobre datos reales porque depende muchísimo del tokenizer que entrenasteis.

100 × 1024 × 3 = 307.200 caracteres

«Para obtener aproximadamente 100 secuencias completas de 1.024 tokens, necesito recoger aproximadamente 307.200 caracteres de código»


# ¿Para qué hacemos todo esto?

Para que el flujo sea eficiente:

```text
CodeParrot (archivos Python)
            │
            ▼
    Recoger ~X caracteres
            │
            ▼
        TOKENIZER
            │
            ▼
    tokens de archivo 1
    tokens de archivo 2
    tokens de archivo 3
            ...
            │
            ▼
    Concatenar con EOS
            │
            ▼
████ EOS ███████ EOS ████ EOS █████████...
            │
            ▼
    Cortar cada 1024 tokens
            │
            ▼
    ┌─────────────┐
    │ 1024 tokens │ → Input 1
    ├─────────────┤
    │ 1024 tokens │ → Input 2
    ├─────────────┤
    │ 1024 tokens │ → Input 3
    ├─────────────┤
    │     ...     │
    └─────────────┘
            │
            ▼
        DataLoader
            │
            ▼
          Batches
            │
            ▼
            GPT
```

In [8]:
#hide_output

# Vamos a estimar "character per token"

from tqdm.auto import tqdm # Para msotrar barra de progreso
from datasets import load_dataset

examples = 500 #vamos a analizar 500 archivos/ejemplos
total_characters = 0 # acumularemos caracteres
total_tokens =  0 #acumularemos tokens



dataset = load_dataset('transformersbook/codeparrot-train', split='train',
                       streaming=True)
# Carga CodeParrot en streaming, porque solo necesitamos ir leyendo ejemplos; no necesitamos descargar todo el dataset.


for _, example in tqdm(zip(range(examples), iter(dataset)), total=examples):
    total_characters += len(example['content'])
    total_tokens += len(tokenizer(example['content']).tokens())

characters_per_token = total_characters / total_tokens

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

```text
range(500)       → 0, 1, 2, ..., 499

iter(dataset)    → archivo1, archivo2, archivo3, ...
                         │
                         ▼
                       zip()
                         │
                         ▼
                  (0, archivo1)
                  (1, archivo2)
                  (2, archivo3)
                       ...
                  (499, archivo500)
```

In [9]:
print(characters_per_token)

3.6231516195736053


En la siguiente celda queremos transformar esto:

```text
documento 1 → longitud variable
documento 2 → longitud variable
documento 3 → longitud variable
...
```

en esto:

```text
tensor de 1024 tokens
tensor de 1024 tokens
tensor de 1024 tokens
...
```

Y hacerlo **sobre la marcha**, sin preparar todo el dataset previamente.

In [17]:
import torch
from torch.utils.data import IterableDataset # clase de PyTorch para crear inputs de constant-length para el modelo

class ConstantLengthDataset(IterableDataset):
    
    def __init__(self, tokenizer, dataset, seq_length=1024,
                 num_of_sequences=1024, chars_per_token=3.6): # guardamos la configuración
        self.tokenizer = tokenizer # tokenizador que entrenamos
        self.concat_token_id = tokenizer.eos_token_id #Obtiene el ID del token: <|endoftext|> q usaremos para separar documentos
        self.dataset = dataset # el codeparrot q estamos leyendo mediante streaming
        self.seq_length = seq_length # longitud de los inputs de entrenamiento (Normalmente 1024 tokens)
        self.input_characters = seq_length * chars_per_token * num_of_sequences #(1024*3.6*1024 = 3.774.874 caracteres
    
    def __iter__(self): # aquí empieza el trabajo real
        iterator = iter(self.dataset) # creamos iterador sobre CodeParrot: documento 1, documento 2....
        more_examples = True
        while more_examples: # crea el buffer # more_examples=True y nunca cambia 
            #Obj: «Cuando termine de procesar un buffer, crea otro buffer y vuelve a hacer todo».
                buffer = [] 
                # almacenará temporalmente documentos como texto
                # Cada vuelta empieza con un buffer vacío                            
                buffer_len = 0 # guarda cuántos caracteres suman esas documentos. Todavía contamos caracteres, no tokens
            
                while True: 
                    # objetivo de while -> seguir metiendo documentos hasta llenar el buffer.
                    if buffer_len >= self.input_characters:
                        m=f"Buffer full: {buffer_len}>={self.input_characters:.0f}"
                        print(m)
                        break # rompe el while True pero no while more_examples:. Sólo termina el proceso de llebar ese buffer
                    try:
                        m=f"Fill buffer: {buffer_len}<{self.input_characters:.0f}"
                        print(m)
                        buffer.append(next(iterator)["content"])
                        # next(iterator) -> dame el siguiente documento de CodeParrot. ->  [content] conrtenido de cada doc
                        buffer_len += len(buffer[-1]) # añadimos en lista la longitud de la última entrada del buffer
                    except StopIteration: # para cuando llegamos al final del documento
                        iterator = iter(self.dataset)
    
                all_token_ids = [] # creamos una lista donde vamos a juntar todos los tokens
                tokenized_inputs = self.tokenizer(buffer, truncation=False) # tokenizamos todo el buffer
                # El tokenizador produce -> tokenized_inputs["input_ids"]
                # [
                # [15, 83, 21, 74, ...],       ← documento 1
                # [91, 14, 53, ...],           ← documento 2
                # [37, 82, 19, 61, ...]        ← documento 3
                # ]
                # Todavía tenemos una lista de tokens por documento
                
                    
                for tokenized_input in tokenized_inputs['input_ids']: # va recorriendo cada doc
                    all_token_ids.extend(tokenized_input + [self.concat_token_id])
                    # all_token_ids.extend() -> Añade a all_token todos los elementos de cada documento (una sola lista, no lista de listas)
                    # (tokenized_input + [self.concat_token_id]) ->concatena los elementos de cada doc + el token EOS, 
                    # que lo ponemos como lista para poder hacer tb .extend() de ese elemento
                    # Resultado ████ doc1 █ EOS █ doc2 █ EOS █ doc3 █ EOS █ doc4 █ EOS █████...
                
    
    
                # Ahora cortamos esta "tira":
            
                for i in range(0, len(all_token_ids), self.seq_length): # de 0 hasta seq:length en saltos de seq_length
                    input_ids = all_token_ids[i : i + self.seq_length]
                    
                    # i = 0
    
                    # all_token_ids[0:1024]
                    #        ↓
                    #     1024 tokens
    
                    # i = 1024
    
                    # all_token_ids[1024:2048]
                    #        ↓
                    #     1024 tokens
                    # Las secuencias de entrenamiento no tienen por qué coincidir con los documentos originales.
                    
                    if len(input_ids) == self.seq_length:
                        yield torch.tensor(input_ids) 
                        # convertimos los input_ids en tensores de pytorch
                        # yield va generando esa secuencia

Ojo! **No necesitamos attention mask porque no estamos haciendo padding** -> Todos los input_ids tienen la misma longhitud (1024)

In [19]:
# Probamos que funciona

shuffled_dataset = dataset.shuffle(buffer_size=100)
# barajamos el dataset
# Como dataset es un dataset en streaming, no puede hacer un shuffle tradicional de todo el dataset: eso requeriría cargar todos los ejemplos.
# En su lugar utiliza un buffer de 100 ejemplos para hacer un barajado aproximado.

constant_length_dataset = ConstantLengthDataset(tokenizer, shuffled_dataset,
                                                num_of_sequences=10)
# Cremos nuestro constant_length_dataset
# tokenizer             → nuestro tokenizer de código
# shuffled_dataset      → CodeParrot barajado
# seq_length            → no lo especificas→ usa 1024
# num_of_sequences=10   → queremos buffers con texto suficiente para unas 10 secuencias. Estamos probando
# chars_per_token       → no lo especificas → usa 3.6



dataset_iterator = iter(constant_length_dataset)

lengths = [len(b) for _, b in zip(range(5), dataset_iterator)]
print(f"Lengths of the sequences: {lengths}")

Fill buffer: 0<36864
Fill buffer: 2550<36864
Fill buffer: 12355<36864
Fill buffer: 14162<36864
Fill buffer: 15953<36864
Fill buffer: 17778<36864
Fill buffer: 20839<36864
Fill buffer: 22080<36864
Fill buffer: 36131<36864
Buffer full: 51975>=36864
Lengths of the sequences: [1024, 1024, 1024, 1024, 1024]


### Defining the Training Loop

#### ¿Quién hace cada cosa durante el entrenamiento?

- `DataLoader` → entrega los **batches**.
- `model(...)` → calcula las **predicciones** y la `loss`.
- `loss.backward()` o `accelerator.backward(loss)` → calcula los **gradientes**.
- `optimizer.step()` → actualiza los **pesos del modelo**.
- `lr_scheduler.step()` → ajusta el **learning rate**.
- `optimizer.zero_grad()` → limpia los gradientes para la siguiente iteración.

```text
DataLoader
    │
    ▼
  Batch
    │
    ▼
model(...)
    │
    ├── predicciones
    │
    └── loss
          │
          ▼
accelerator.backward(loss)
          │
          ▼
     gradientes
          │
          ▼
  optimizer.step()
          │
          ▼
 actualizar pesos
          │
          ▼
lr_scheduler.step()
          │
          ▼
ajustar learning rate
          │
          ▼
optimizer.zero_grad()
          │
          ▼
 siguiente batch
```

In [20]:
# Esta celda todavía no entrena nada. Simplemente reúne en args todos los hiperparámetros 
# que utilizará después el training loop.


from argparse import Namespace


# Commented parameters correspond to the small model
config = {"train_batch_size": 2, # 12
          "valid_batch_size": 2, # 12
          "weight_decay": 0.1,
          "shuffle_buffer": 1000,
          "learning_rate": 2e-4, # 5e-4
          "lr_scheduler_type": "cosine",
          "num_warmup_steps": 750, # 2000
          "gradient_accumulation_steps": 16, # 1
          "max_train_steps": 50000, # 150000
          "max_eval_steps": -1,
          "seq_length": 1024,
          "seed": 1,
          "save_checkpoint_steps": 50000} # 15000

args = Namespace(**config)

# Namespace -> un contenedor sencillo que permite guardar valores como atributos.
# Tienes este diccionario:
# config = {
#     "train_batch_size": 2,
#    "learning_rate": 2e-4,
#    "seq_length": 1024
#}

# Normalmente accedería así:

    # config["train_batch_size"]
    # config["learning_rate"]

# Pero como hacemos Namespace(**config) -> podemos escribir:
    # args.train_batch_size
    # args.learning_rate
    # args.seq_length


In [ ]:
# Esta función no entrena el modelo. Prepara todo el sistema de registro y seguimiento del entrenamiento: 
# guardar mensajes, métricas e hiperparámetros para poder ver después qué ocurrió.


from torch.utils.tensorboard import SummaryWriter 
# escribe información que luego puede visualizar TensorBoard: loss, learning rate, hiperparámetros, etc.
import logging 
# es de Python y sirve para registrar mensajes
# en setup_logging -> para mensajes del programa
import wandb
# es Weights & Biases, otra plataforma de seguimiento de experimentos. 
# Hace algo parecido a TensorBoard pero con una interfaz web y gestión de experimentos.
# en setup_logging -> para seguimiento online del experimento


def setup_logging(project_name):
    logger = logging.getLogger(__name__)
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S", level=logging.INFO, handlers=[
        logging.FileHandler(f"log/debug_{accelerator.process_index}.log"),
        logging.StreamHandler()])
    if accelerator.is_main_process: # We only want to set up logging once
        wandb.init(project=project_name, config=args)
        run_name = wandb.run.name
        tb_writer = SummaryWriter()
        tb_writer.add_hparams(vars(args), {'0': 0})
        logger.setLevel(logging.INFO)
        datasets.utils.logging.set_verbosity_debug()
        transformers.utils.logging.set_verbosity_info()
    else:
        tb_writer = None
        run_name = ''
        logger.setLevel(logging.ERROR)
        datasets.utils.logging.set_verbosity_error()
        transformers.utils.logging.set_verbosity_error()
    return logger, tb_writer, run_name

In [ ]:
def log_metrics(step, metrics):
    logger.info(f"Step {step}: {metrics}")
    if accelerator.is_main_process:
        wandb.log(metrics)
        [tb_writer.add_scalar(k, v, step) for k, v in metrics.items()]

In [ ]:
#hide_output
from torch.utils.data.dataloader import DataLoader

def create_dataloaders(dataset_name):
    train_data = load_dataset(dataset_name+'-train', split="train",
                              streaming=True)
    train_data = train_data.shuffle(buffer_size=args.shuffle_buffer,
                                    seed=args.seed)
    valid_data = load_dataset(dataset_name+'-valid', split="validation",
                              streaming=True)
    
    train_dataset = ConstantLengthDataset(tokenizer, train_data,
                                          seq_length=args.seq_length)
    valid_dataset = ConstantLengthDataset(tokenizer, valid_data,
                                          seq_length=args.seq_length)
    
    train_dataloader=DataLoader(train_dataset, batch_size=args.train_batch_size)
    eval_dataloader=DataLoader(valid_dataset, batch_size=args.valid_batch_size)
    return train_dataloader, eval_dataloader

In [ ]:
def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [{'params': params_with_wd, 'weight_decay': args.weight_decay},
            {'params': params_without_wd, 'weight_decay': 0.0}]

In [ ]:
def evaluate():
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(batch, labels=batch)
        loss = outputs.loss.repeat(args.valid_batch_size)
        losses.append(accelerator.gather(loss))
        if args.max_eval_steps > 0 and step >= args.max_eval_steps: break
    loss = torch.mean(torch.cat(losses))
    try:
        perplexity = torch.exp(loss)
    except OverflowError:
        perplexity = torch.tensor(float("inf"))
    return loss.item(), perplexity.item()

In [ ]:
set_seed(args.seed)

# Accelerator
accelerator = Accelerator()
samples_per_step = accelerator.state.num_processes * args.train_batch_size

# Logging
logger, tb_writer, run_name = setup_logging(project_name.split("/")[1])
logger.info(accelerator.state)

# Load model and tokenizer
if accelerator.is_main_process:
    hf_repo = Repository("./", clone_from=project_name, revision=run_name)
model = AutoModelForCausalLM.from_pretrained("./", gradient_checkpointing=True)
tokenizer = AutoTokenizer.from_pretrained("./")

# Load dataset and dataloader
train_dataloader, eval_dataloader = create_dataloaders(dataset_name)

# Prepare the optimizer and learning rate scheduler
optimizer = AdamW(get_grouped_params(model), lr=args.learning_rate)
lr_scheduler = get_scheduler(name=args.lr_scheduler_type, optimizer=optimizer,
                             num_warmup_steps=args.num_warmup_steps,
                             num_training_steps=args.max_train_steps,)
def get_lr():
    return optimizer.param_groups[0]['lr']

# Prepare everything with our `accelerator` (order of args is not important)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader)

# Train model
model.train()
completed_steps = 0
for step, batch in enumerate(train_dataloader, start=1):
    loss = model(batch, labels=batch).loss
    log_metrics(step, {'lr': get_lr(), 'samples': step*samples_per_step,
                       'steps': completed_steps, 'loss/train': loss.item()})
    loss = loss / args.gradient_accumulation_steps
    accelerator.backward(loss)
    if step % args.gradient_accumulation_steps == 0:
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        completed_steps += 1
    if step % args.save_checkpoint_steps == 0:
        logger.info('Evaluating and saving model checkpoint')
        eval_loss, perplexity = evaluate()
        log_metrics(step, {'loss/eval': eval_loss, 'perplexity': perplexity})
        accelerator.wait_for_everyone()
        unwrapped_model = accelerator.unwrap_model(model)
        if accelerator.is_main_process:
            unwrapped_model.save_pretrained("./")
            hf_repo.push_to_hub(commit_message=f'step {step}')
        model.train()
    if completed_steps >= args.max_train_steps:
        break

# Evaluate and save the last checkpoint
logger.info('Evaluating and saving model after training')
eval_loss, perplexity = evaluate()
log_metrics(step, {'loss/eval': eval_loss, 'perplexity': perplexity})
accelerator.wait_for_everyone()
unwrapped_model = accelerator.unwrap_model(model)
if accelerator.is_main_process:
    unwrapped_model.save_pretrained("./")
    hf_repo.push_to_hub(commit_message=f'final model')

<img alt="DDP" caption="Illustration of the processing steps in DDP with four GPUs" src="images/chapter10_ddp.png" id="ddp"/>

### The Training Run

## Results and Analysis

In [ ]:
#hide_output
from transformers import pipeline, set_seed

model_ckpt = 'transformersbook/codeparrot-small'
generation = pipeline('text-generation', model=model_ckpt, device=0)

In [ ]:
import re
from transformers import set_seed 

def first_block(string):
    return re.split('\nclass|\ndef|\n#|\n@|\nprint|\nif', string)[0].rstrip()

def complete_code(pipe, prompt, max_length=64, num_completions=4, seed=1):
    set_seed(seed)
    gen_kwargs = {"temperature":0.4, "top_p":0.95, "top_k":0, "num_beams":1,
                  "do_sample":True,}
    code_gens = generation(prompt, num_return_sequences=num_completions, 
                            max_length=max_length, **gen_kwargs)
    code_strings = []
    for code_gen in code_gens:
        generated_code = first_block(code_gen['generated_text'][len(prompt):])
        code_strings.append(generated_code)
    print(('\n'+'='*80 + '\n').join(code_strings))

In [ ]:
prompt = '''def area_of_rectangle(a: float, b: float):
    """Return the area of the rectangle."""'''
complete_code(generation, prompt)

In [ ]:
prompt = '''def get_urls_from_html(html):
    """Get all embedded URLs in a HTML string."""'''
complete_code(generation, prompt)

In [ ]:
import requests

def get_urls_from_html(html):
    return [url for url in re.findall(r'<a href="(.*?)"', html) if url]

print(" | ".join(get_urls_from_html(requests.get('https://hf.co/').text)))

> **NOTE**: In the following code block, a large GPT-2 checkpoint is loaded into memory. On platforms like Colab and Kaggle, this can cause the instance to crash due to insufficient RAM or GPU memory. You can still run the example if you replace the large model with the small one by using `model_ckpt = "transformersbook/codeparrot-small"`.
 

In [ ]:
model_ckpt = 'transformersbook/codeparrot'
generation = pipeline('text-generation', model=model_ckpt, device=0)

prompt = '''# a function in native python:
def mean(a):
    return sum(a)/len(a)

# the same function using numpy:
import numpy as np
def mean(a):'''
complete_code(generation, prompt, max_length=64)

In [ ]:
prompt = '''X = np.random.randn(100, 100)
y = np.random.randint(0, 1, 100)

# fit random forest classifier with 20 estimators'''
complete_code(generation, prompt, max_length=96)

## Conclusion